# 04 · Results (run any time; works on partial results)
Writes tables to `tables/` and figures to `figures/` in the shared folder.

In [ ]:
# ---- setup: identical in every notebook ----
FOLDER = "AMR_AL"          # name of the shared folder (or its shortcut) in My Drive

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
ROOT = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(os.path.join(ROOT, "amr_al")):
    raise SystemExit(f"{ROOT}/amr_al not found.\nIn Google Drive open 'Shared with me', right-click {FOLDER} -> "
                     "Organize -> Add shortcut -> My Drive, then run this cell again.")
try:
    import xgboost
    if int(xgboost.__version__.split(".")[0]) < 2:
        raise ImportError
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
os.environ["AMR_AL_ROOT"] = ROOT
sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith("amr_al")]:
    del sys.modules[m]                      # always load the latest code from Drive
from amr_al import config, paths, data, models, runner, audit, analysis, driams_fetch
paths.ensure_dirs()
print("shared folder:", ROOT)
print("compute device for XGBoost:", models.device())

In [ ]:
runner.status()

In [ ]:
from IPython.display import Image, display
for exp in config.PRIORITY:
    out = analysis.report(exp)
    if not out:
        continue
    if "gap" in out and len(out["gap"]):
        print(exp, "- labels needed to close 95% of the gap between no retraining and full retraining:")
        display(out["gap"].round(3))
    if len(out.get("tests", [])):
        print(exp, "- strategy vs random (paired over seeds, Holm-corrected):")
        display(out["tests"].round(4))
    for f in out["figs"]:
        display(Image(f))

Tables are in `tables/` and figures in `figures/` of the shared folder.